# 03 Scoring And H3

本 notebook 从 01、02 生成的缓存数据出发，完成上海 15 分钟城市宜居性评价的最终评分与 H3 汇聚。流程包括：读取 500 米网格与四种出行方式的可达计数，计算基础层得分，叠加 Track A — Healthy Lifestyle & Sport 健康生活与运动赛道指标，加入绿地与 AQI 环境健康代理变量，汇聚至 Uber H3 resolution 8 六边形，并导出 GeoJSON、Parquet 和摘要 JSON。

评分结果面向两个用途：一是作为课程分析中“基础层 + 赛道层”的综合评价结果；二是作为公众网页应用的空间数据底图，支持按步行、自行车、公交、驾车四种方式理解不同区域的健康生活方式可达性。


## 1. 环境与缓存读取

本节读取前两个 notebook 生成的中间缓存。若首次运行，请先依次运行 `01_data_collection.ipynb` 和 `02_grid_isochrones.ipynb`，生成 `grid_500m.gpkg`、`grid_access_counts.parquet` 和绿地缓存。


In [ ]:
from pathlib import Path
import json

import geopandas as gpd
import h3
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

SUBMIT_DIR = ROOT / "submit"
CACHE_DIR = SUBMIT_DIR / "cache"
OUTPUT_DIR = SUBMIT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_GPKG = CACHE_DIR / "grid_500m.gpkg"
ACCESS_PARQUET = CACHE_DIR / "grid_access_counts.parquet"
GREEN_GPKG = CACHE_DIR / "ai_green_all.gpkg"
AQI_DISTRICT_CSV = (
    ROOT
    / "data"
    / "shanghai_aqi_official_2025_20250101_20251231_package"
    / "clean"
    / "shanghai_aqi_district_daily_sthj_official_20250101_20251231.csv"
)
AQI_CITY_CSV = (
    ROOT
    / "data"
    / "shanghai_aqi_official_2025_20250101_20251231_package"
    / "clean"
    / "shanghai_aqi_city_daily_sthj_official_20250101_20251231.csv"
)

required = [GRID_GPKG, ACCESS_PARQUET]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("请先运行 01、02 notebook 生成缓存文件：" + "; ".join(missing))

grid = gpd.read_file(GRID_GPKG, layer="grid").to_crs("EPSG:4326")
centers = gpd.read_file(GRID_GPKG, layer="grid_centers").to_crs("EPSG:4326")
access = pd.read_parquet(ACCESS_PARQUET)

for col in ["center_lon", "center_lat"]:
    if col in grid.columns:
        grid = grid.drop(columns=col)
center_cols = [col for col in ["grid_id", "center_lon", "center_lat"] if col in centers.columns]
grid = grid.merge(centers[center_cols], on="grid_id", how="left")
grid = grid.merge(access, on="grid_id", how="left")

grid[["grid_id", "district_name", "town_name", "center_lon", "center_lat"]].head()


## 2. 评分权重

基础层衡量日常生活服务的综合通达性，使用步行 15 分钟内可达设施数量作为主要口径。Track A 健康生活与运动赛道聚焦健身房、户外健身点、运动场地、公共游泳池、瑜伽/武术/舞蹈工作室、新鲜市场与健康食品零售，同时叠加绿地和 AQI。单项得分采用上限截断：`score = min(count / target, 1) * 100`。


In [ ]:
BASE_WEIGHTS = {
    "education": {"label": "教育", "mode": "walk", "target": 3, "weight": 0.15},
    "healthcare": {"label": "医疗", "mode": "walk", "target": 2, "weight": 0.15},
    "shopping": {"label": "购物", "mode": "walk", "target": 5, "weight": 0.15},
    "transport": {"label": "公共交通", "mode": "walk", "target": 3, "weight": 0.15},
    "daily_service": {"label": "日常服务", "mode": "walk", "target": 5, "weight": 0.15},
    "leisure": {"label": "休闲", "mode": "walk", "target": 2, "weight": 0.10},
    "public_service": {"label": "公共服务", "mode": "walk", "target": 2, "weight": 0.10},
    "food": {"label": "餐饮食品", "mode": "walk", "target": 5, "weight": 0.05},
}

TRACK_A_WEIGHTS = {
    "fitness": {"label": "健身房/健身工作室", "mode": "bike", "target": 1, "weight": 0.20},
    "outdoor_fitness": {"label": "户外健身器材", "mode": "walk", "target": 1, "weight": 0.13},
    "sport_fields": {"label": "跑道/运动场/篮球场", "mode": "walk", "target": 1, "weight": 0.20},
    "public_pool": {"label": "公共游泳池", "mode": "walk", "target": 1, "weight": 0.13},
    "mind_body": {"label": "瑜伽/武术/舞蹈工作室", "mode": "walk", "target": 1, "weight": 0.14},
    "fresh_food": {"label": "新鲜市场/健康食品零售", "mode": "walk", "target": 2, "weight": 0.20},
}

TRACK_A_COMPONENT_WEIGHTS = {
    "sport_facility_score": 0.72,
    "aqi_score": 0.18,
    "green_score": 0.10,
}
COMBINED_WEIGHTS = {
    "base_score": 0.60,
    "track_a_score": 0.40,
}
GREEN_TARGET = 0.20
H3_RESOLUTION = 8

weight_rows = []
for group_name, weights in [("base", BASE_WEIGHTS), ("track_a_facility", TRACK_A_WEIGHTS)]:
    for key, cfg in weights.items():
        weight_rows.append({"group": group_name, "indicator": key, **cfg})

pd.DataFrame(weight_rows)


## 3. 绿地与 AQI 环境指标

绿地使用 AI 解译绿地面与 500 米网格相交面积计算比例，达到 20% 绿地覆盖视为满分。AQI 使用区级逐日数据计算年均值和达标天数占比，AQI 越低得分越高；若某网格缺少区级匹配，则回退到全市平均。


In [ ]:
grid_proj = grid.to_crs("EPSG:32651")
grid_area = grid_proj.set_index("grid_id").geometry.area

if GREEN_GPKG.exists():
    green = gpd.read_file(GREEN_GPKG, layer="green").to_crs("EPSG:32651")
    if not green.empty:
        overlay = gpd.overlay(
            grid_proj[["grid_id", "geometry"]],
            green[["geometry"]],
            how="intersection",
            keep_geom_type=False,
        )
        if not overlay.empty:
            overlay["green_area"] = overlay.geometry.area
            green_area = overlay.groupby("grid_id")["green_area"].sum()
            grid["green_ratio"] = grid["grid_id"].map(green_area).fillna(0) / grid["grid_id"].map(grid_area)
        else:
            grid["green_ratio"] = 0.0
    else:
        grid["green_ratio"] = 0.0
else:
    grid["green_ratio"] = 0.0

grid["green_ratio"] = grid["green_ratio"].fillna(0).clip(0, 1)
grid["green_score"] = (grid["green_ratio"] / GREEN_TARGET).clip(upper=1) * 100

def aqi_to_score(aqi):
    if pd.isna(aqi):
        return np.nan
    return float(np.clip((100 - float(aqi)) / 65 * 100, 0, 100))

city_aqi_avg = np.nan
city_good_days_pct = np.nan
if AQI_CITY_CSV.exists():
    city_aqi = pd.read_csv(AQI_CITY_CSV)
    city_aqi["aqi"] = pd.to_numeric(city_aqi["aqi"], errors="coerce")
    city_aqi_avg = float(city_aqi["aqi"].mean())
    city_good_days_pct = float((city_aqi["aqi"] <= 100).mean() * 100)

if AQI_DISTRICT_CSV.exists() and "district_name" in grid.columns:
    aqi_dist = pd.read_csv(AQI_DISTRICT_CSV)
    aqi_dist["aqi"] = pd.to_numeric(aqi_dist["aqi"], errors="coerce")
    aqi_summary = (
        aqi_dist.groupby("group_name")
        .agg(
            aqi_avg=("aqi", "mean"),
            aqi_good_days_pct=("aqi", lambda s: (s <= 100).mean() * 100),
        )
        .reset_index()
        .rename(columns={"group_name": "district_name"})
    )
    aqi_summary["aqi_score"] = aqi_summary["aqi_avg"].map(aqi_to_score)
    grid = grid.merge(aqi_summary, on="district_name", how="left")
else:
    grid["aqi_avg"] = np.nan
    grid["aqi_good_days_pct"] = np.nan
    grid["aqi_score"] = np.nan

grid["aqi_avg"] = grid["aqi_avg"].fillna(city_aqi_avg)
grid["aqi_good_days_pct"] = grid["aqi_good_days_pct"].fillna(city_good_days_pct)
grid["aqi_score"] = grid["aqi_score"].fillna(aqi_to_score(city_aqi_avg)).fillna(50)

grid[["grid_id", "green_ratio", "green_score", "aqi_avg", "aqi_score", "aqi_good_days_pct"]].head()


## 4. 网格层基础分与赛道分

基础层与 Track A 设施层先在 500 米网格上计算。Track A 最终分由运动健康设施可达性、AQI 和绿地共同构成，综合分再按基础层 60%、Track A 40% 汇总。


In [ ]:
def capped_score(series, target):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return (values / target).clip(upper=1) * 100

for key, cfg in BASE_WEIGHTS.items():
    count_col = f"{cfg['mode']}__{key}"
    if count_col not in grid.columns:
        grid[count_col] = 0
    grid[f"base__{key}__score"] = capped_score(grid[count_col], cfg["target"])

for key, cfg in TRACK_A_WEIGHTS.items():
    count_col = f"{cfg['mode']}__{key}"
    if count_col not in grid.columns:
        grid[count_col] = 0
    grid[f"tracka__{key}__score"] = capped_score(grid[count_col], cfg["target"])

grid["base_score"] = sum(grid[f"base__{key}__score"] * cfg["weight"] for key, cfg in BASE_WEIGHTS.items())
grid["sport_facility_score"] = sum(grid[f"tracka__{key}__score"] * cfg["weight"] for key, cfg in TRACK_A_WEIGHTS.items())
grid["track_a_score"] = (
    grid["sport_facility_score"] * TRACK_A_COMPONENT_WEIGHTS["sport_facility_score"]
    + grid["aqi_score"] * TRACK_A_COMPONENT_WEIGHTS["aqi_score"]
    + grid["green_score"] * TRACK_A_COMPONENT_WEIGHTS["green_score"]
)
grid["combined_score"] = (
    grid["base_score"] * COMBINED_WEIGHTS["base_score"]
    + grid["track_a_score"] * COMBINED_WEIGHTS["track_a_score"]
)

score_preview_cols = ["base_score", "sport_facility_score", "green_score", "aqi_score", "track_a_score", "combined_score"]
grid[score_preview_cols].describe().round(2)


## 5. H3 汇聚与陆地过滤

网格中心点映射到 H3 resolution 8 六边形后，对得分取平均、对设施计数求和。若已有运动荒漠脚本生成的陆地 H3 列表，则用该列表过滤海域和江面单元，避免行政区边界中的水域影响最终地图。


In [ ]:
grid["h3"] = [h3.latlng_to_cell(float(lat), float(lon), H3_RESOLUTION) for lat, lon in zip(grid["center_lat"], grid["center_lon"])]

score_columns = [
    "base_score", "sport_facility_score", "green_ratio", "green_score",
    "aqi_avg", "aqi_score", "aqi_good_days_pct", "track_a_score", "combined_score",
]
facility_count_cols = [f"{cfg['mode']}__{key}" for key, cfg in TRACK_A_WEIGHTS.items()]
facility_count_cols = [col for col in facility_count_cols if col in grid.columns]

aggregation = {col: "mean" for col in score_columns}
aggregation.update({col: "sum" for col in facility_count_cols})

if "district_name" in grid.columns:
    aggregation["district_name"] = lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else None
if "town_name" in grid.columns:
    aggregation["town_name"] = lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else None

h3_scores = grid.groupby("h3", as_index=False).agg(aggregation)
h3_scores["cell_count"] = grid.groupby("h3").size().reindex(h3_scores["h3"]).to_numpy()

land_h3_path = ROOT / "outputs" / "sports_desert" / "sports_desert_h3.parquet"
land_filter_note = "未应用陆地过滤"
if land_h3_path.exists():
    land_h3 = set(pd.read_parquet(land_h3_path, columns=["h3"])["h3"].astype(str))
    before = len(h3_scores)
    h3_scores = h3_scores[h3_scores["h3"].astype(str).isin(land_h3)].reset_index(drop=True)
    land_filter_note = f"应用陆地过滤：{before} -> {len(h3_scores)} 个 H3"

land_filter_note, h3_scores.head()


## 6. 运动荒漠分类

根据 Track A 得分划分运动健康资源薄弱程度：低于 25 为 Severe Desert，25–40 为 Desert，40–55 为 Vulnerable，55 以上为 Adequate。该分类用于识别体育与健康生活方式资源连续低值区。


In [ ]:
def classify_sports_desert(score):
    if score < 25:
        return "Severe Desert"
    if score < 40:
        return "Desert"
    if score < 55:
        return "Vulnerable"
    return "Adequate"

h3_scores["sports_desert_class"] = h3_scores["track_a_score"].map(classify_sports_desert)
h3_scores["is_sports_desert"] = h3_scores["sports_desert_class"].isin(["Severe Desert", "Desert"])

h3_scores["sports_desert_class"].value_counts().rename_axis("class").reset_index(name="h3_count")


## 7. GeoJSON、Parquet 与摘要导出

最终输出包括三个文件：`03_h3_scores.geojson` 用于地图展示，`03_h3_scores.parquet` 用于后续分析，`03_h3_summary.json` 用于快速核对总体指标。GeoJSON 导出时显式清理 NaN，避免浏览器端 JSON 解析失败。


In [ ]:
def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float) and (np.isnan(value) or np.isinf(value)):
        return None
    if pd.isna(value):
        return None
    return value

features = []
for _, row in h3_scores.iterrows():
    boundary = h3.cell_to_boundary(row["h3"])
    coordinates = [[(lng, lat) for lat, lng in boundary] + [(boundary[0][1], boundary[0][0])]]
    properties = {key: json_safe(value) for key, value in row.drop(labels=["h3"]).to_dict().items()}
    properties["h3"] = row["h3"]
    features.append({
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": coordinates},
        "properties": properties,
    })

geojson = {"type": "FeatureCollection", "features": features}
geojson_path = OUTPUT_DIR / "03_h3_scores.geojson"
parquet_path = OUTPUT_DIR / "03_h3_scores.parquet"
summary_path = OUTPUT_DIR / "03_h3_summary.json"

geojson_path.write_text(json.dumps(geojson, ensure_ascii=False, allow_nan=False), encoding="utf-8")
h3_scores.to_parquet(parquet_path, index=False)

summary = {
    "h3_resolution": H3_RESOLUTION,
    "h3_count": int(len(h3_scores)),
    "base_score_mean": round(float(h3_scores["base_score"].mean()), 2),
    "track_a_score_mean": round(float(h3_scores["track_a_score"].mean()), 2),
    "combined_score_mean": round(float(h3_scores["combined_score"].mean()), 2),
    "sports_desert_rate": round(float(h3_scores["is_sports_desert"].mean()), 4),
    "sports_desert_distribution": {k: int(v) for k, v in h3_scores["sports_desert_class"].value_counts().to_dict().items()},
    "outputs": {
        "geojson": str(geojson_path.relative_to(ROOT)),
        "parquet": str(parquet_path.relative_to(ROOT)),
        "summary": str(summary_path.relative_to(ROOT)),
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

summary


## 8. 结果预览

下表列出综合得分最高的 H3 单元，并保留行政区、街镇、基础层得分、Track A 得分和运动荒漠分类，便于与网页端点击详情进行核对。


In [ ]:
preview_cols = [
    "h3", "district_name", "town_name", "base_score", "track_a_score",
    "combined_score", "green_ratio", "aqi_avg", "sports_desert_class",
]
preview_cols = [col for col in preview_cols if col in h3_scores.columns]
h3_scores.sort_values("combined_score", ascending=False)[preview_cols].head(10)
